In [ ]:
%matplotlib inline
%reset -f

import os, random, time, sys, warnings

import numpy as np
import scipy as sp
import pandas as pd
from tqdm import tqdm
import bct

import torch

from src.utils import get_weight_masks_schaefer, get_file_str

# import plotting libraries
import matplotlib.pyplot as plt
import matplotlib.patches as patches
plt.rcParams.update({"font.size": 10})
plt.rcParams["svg.fonttype"] = "none"
plt.rc('font', family='DejaVu Sans')
import seaborn as sns
sns.set_style("white")
from src.plotting import my_reg_plot

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

In [ ]:
# Device configuration
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cpu')

In [ ]:
save_figs = True

In [ ]:
# directories
datadir = '/home/lindenmp/research_projects/neuro_rnn/data'
modeldir = '/media/lindenmp/storage_ssd/research_projects/neuro_rnn/results/pytorch/model'
outdir = '/home/lindenmp/research_projects/neuro_rnn/results/figs'

# data parameters
task = 'PerceptualDecisionMaking-v0'
# task = 'MultiSensoryIntegration-v0'
# task = 'ContextDecisionMaking-v0'
dt = 100
batch_size = 32
decision = 300
if task == 'PerceptualDecisionMaking-v0':
    seq_len = 22
elif task == 'MultiSensoryIntegration-v0':
    seq_len = 11
elif task == 'ContextDecisionMaking-v0':
    seq_len = 13
seq_len = seq_len + int((decision - 100) / dt)
seq_len_multi = 5
seq_len = seq_len * seq_len_multi
print(seq_len)

# RNN model and training parameters
rnn_model = 'rnn-tanh'
hidden_size = 100
n_runs = 25
n_epochs = 30000
lr = 0.001

# regularization parameters
reg_type = 'l2'
reg_weight = 0.0015
mask_weights = True
kernel_type = 'sa_axis'
# kernel_type = 'euclidean'
# kernel_type = None

In [ ]:
timing = {'fixation': 200, 'stimulus': 1000, 'delay': 0, 'decision': decision}
env_kwargs = {'dt': dt, 'timing': timing}

config = {
    'datadir': datadir, 'outdir': outdir,
    'task': task, 'dt': dt, 'seq_len': seq_len, 'batch_size': batch_size,  # data parameters
    'rnn_model': rnn_model, 'hidden_size': hidden_size, 'n_runs': n_runs, 'n_epochs': n_epochs, 'lr': lr, 'mask_weights': mask_weights,  # RNN model and training parameters
    'reg_type': reg_type, 'reg_weight': reg_weight, 'kernel_type': kernel_type, # regularization parameters
    'env_kwargs': env_kwargs,
}

In [ ]:
# weight masks
if mask_weights:
    centroids = pd.read_csv(os.path.join(datadir, 'schaefer{0}_centroids.csv'.format(hidden_size * 2)))
    centroids = centroids[:hidden_size]
    roi_names = list(centroids['ROI Name'])
    input_system = 'Vis'
    output_system = 'Default'
    masks = get_weight_masks_schaefer(roi_names=roi_names, input_system=input_system, output_system=output_system)
    n_io = '{0}-{1}'.format(np.sum(masks['input_weight_mask']), np.sum(masks['output_weight_mask']))
else:
    n_io = 'na'
config['n_io'] = n_io

file_str = get_file_str(config)
print(file_str)

In [ ]:
# load data
log_args = np.load(os.path.join(modeldir, file_str + '.npy'), allow_pickle=True).item()
training_loss = log_args['training_loss']
validation_loss = log_args['validation_loss']
test_accuracy = log_args['test_accuracy']
n_logged_epochs = test_accuracy.shape[-1]
x_step = int(n_epochs / (n_logged_epochs-1))

checkpoint = torch.load(os.path.join(modeldir, file_str + '.pt'), map_location=device)
hidden_weights = np.zeros((n_runs, n_logged_epochs, hidden_size, hidden_size))
for i, run in enumerate(checkpoint.keys()):
    for j, epoch in enumerate(checkpoint[run].keys()):
        hidden_weights[i, j] = checkpoint[run][epoch]['rnn.weight_hh_l0']
del checkpoint

test_accuracy = test_accuracy[:, :51]
hidden_weights = hidden_weights[:, :51]
n_logged_epochs = test_accuracy.shape[-1]
n_epochs = (test_accuracy.shape[1] - 1) * 100
x_step = int(n_epochs / (n_logged_epochs-1))

print(training_loss.shape)
print(validation_loss.shape)
print(test_accuracy.shape)
print(hidden_weights.shape)
print(n_logged_epochs, n_epochs)

# Model performance vs. degree

In [ ]:
degree_skewness = np.zeros((n_runs, n_logged_epochs))
degree_max = np.zeros((n_runs, n_logged_epochs))
degree_trimmed_mean = np.zeros((n_runs, n_logged_epochs))
rich_club_coef = np.zeros((n_runs, n_logged_epochs))

for run in tqdm(np.arange(n_runs)):
    for epoch in np.arange(n_logged_epochs):
        A = hidden_weights[run, epoch].copy()
        A = np.abs(A)
        np.fill_diagonal(A, 0)
        thresh = np.quantile(A, q=0.9)
        mask = A > thresh
        A[mask] = 1
        A[~mask] = 0

        _, _, degree = bct.degrees_dir(A)
        x, y = np.unique(degree, return_counts=True)
        degree_skewness[run, epoch] = sp.stats.skew(y)
        degree_max[run, epoch] = np.nanmax(x)

        thresh = np.quantile(degree, q=0.9)
        degree_trimmed_mean[run, epoch] = np.mean(degree[degree>thresh])

        R, Nk, Ek = bct.rich_club_bd(A)
        rich_club_coef[run, epoch] = np.nanmax(R)

# Plot

In [ ]:
fig_width = 8.5
fig_height = 8
f = plt.figure(layout=None, figsize=(fig_width, fig_height))
gs = f.add_gridspec(nrows=4, ncols=4, hspace=0.75, wspace=0.2)
ax0_0 = f.add_subplot(gs[0, 0])
ax0_1 = f.add_subplot(gs[0, 1])
ax0_2 = f.add_subplot(gs[0, 2])
ax0_3 = f.add_subplot(gs[0, 3])
ax1 = f.add_subplot(gs[1, :])
ax1_twin = ax1.twinx()
ax2_0 = f.add_subplot(gs[2, 0])
ax2_1 = f.add_subplot(gs[2, 1])
ax2_2 = f.add_subplot(gs[2, 2])
ax2_3 = f.add_subplot(gs[2, 3])
ax3 = f.add_subplot(gs[3, :])
ax3_twin = ax3.twinx()

# rows 0 and 2
# indices = [3, 10, 50, 80]
# indices = [5, 15, 25, 45]
indices = np.linspace(2.5, n_logged_epochs-10, 4).astype(int)
n_indices = len(indices)
color_palette = sns.color_palette("mako", n_colors=n_runs)
ax0_list = [ax0_0, ax0_1, ax0_2, ax0_3]
ax2_list = [ax2_0, ax2_1, ax2_2, ax2_3]
for run in np.arange(n_runs):
    for epoch in np.arange(n_indices):
        A = hidden_weights[run, indices[epoch]].copy()
        A = np.abs(A)
        np.fill_diagonal(A, 0)
        thresh = np.quantile(A, q=0.9)
        mask = A > thresh
        A[mask] = 1
        A[~mask] = 0

        _, _, degree = bct.degrees_dir(A)
        x, y = np.unique(degree, return_counts=True)
        ax0_list[epoch].bar(x, y, alpha=0.25, color=color_palette[run])
        ax0_list[epoch].set_title('Epoch: {0}'.format(indices[epoch]*100))
        ax0_list[epoch].set_xlabel("Node Degree, k")

        R, Nk, Ek = bct.rich_club_bd(A)
        ax2_list[epoch].plot(np.arange(0, len(R)), R, alpha=0.25, color=color_palette[run], linewidth=1)
        ax2_list[epoch].set_xlabel("Node Degree, k")

        if epoch == 0:
            ax2_list[epoch].set_ylabel("Rich-club coefficient")
            ax0_list[epoch].set_ylabel("# of Nodes")

# set axis limits and despine
for epoch in np.arange(n_indices):
    ax0_list[epoch].set_xlim([0, 85])
    ax2_list[epoch].set_xlim([0, 85])
    ax0_list[epoch].set_ylim([0, 11])
    ax2_list[epoch].set_ylim([0, 1])
    sns.despine(offset=5, trim=True, left=False, right=True, top=True, bottom=False, ax=ax0_list[epoch])
    sns.despine(offset=5, trim=True, left=False, right=True, top=True, bottom=False, ax=ax2_list[epoch])

# rows 1 and 3
color_palette = sns.color_palette("Set2")
x_step = int(n_epochs / (n_logged_epochs-1))
x = np.arange(x_step, n_epochs + x_step, x_step)

# add accuracy to each plot
accuracy_mean = test_accuracy.mean(axis=0)[1:]*100
accuracy_std = test_accuracy.std(axis=0)[1:]*100
ci = 1.96 * (accuracy_std / np.sqrt(n_runs))
accuracy_ci_lower = accuracy_mean - ci
accuracy_ci_upper = accuracy_mean + ci

for this_ax in [ax1, ax3]:
    this_ax.plot(x, accuracy_mean, color=color_palette[0], label='RNN-SA')
    this_ax.fill_between(x, accuracy_ci_lower, accuracy_ci_upper, color=color_palette[0], alpha=0.25)
    this_ax.set_ylim([-2.5, 100])
    this_ax.set_ylabel('Accuracy (%)')
    this_ax.set_xlabel('Epochs')
    for epoch in np.arange(n_indices):
        this_ax.axvline(indices[epoch]*100, color='gray', linestyle='--')

# Degree, trimmed mean
y_mean = degree_trimmed_mean.mean(axis=0)[1:]
y_std = degree_trimmed_mean.std(axis=0)[1:]
ci = 1.96 * (y_std / np.sqrt(n_runs))
ci_lower = y_mean - ci
ci_upper = y_mean + ci

ax1_twin.plot(x, y_mean, color='k', label='Degree')
ax1_twin.fill_between(x, ci_lower, ci_upper, color='k', alpha=0.25)
ax1_twin.set_ylabel('Node Degree (mean)')
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()
ax1_twin.legend(lines + lines2, labels + labels2, loc='lower right')

# Rich club coefficient
y_mean = rich_club_coef.mean(axis=0)[1:]
y_std = rich_club_coef.std(axis=0)[1:]
ci = 1.96 * (y_std / np.sqrt(n_runs))
ci_lower = y_mean - ci
ci_upper = y_mean + ci

ax3_twin.plot(x, y_mean, color='k', label='Rich-club coefficient')
ax3_twin.fill_between(x, ci_lower, ci_upper, color='k', alpha=0.25)
ax3_twin.set_ylabel('RC (max)')
lines, labels = ax3.get_legend_handles_labels()
lines2, labels2 = ax3_twin.get_legend_handles_labels()
ax3_twin.legend(lines + lines2, labels + labels2, loc='lower right')

# set axis limits and despine
for this_ax in [ax1, ax3, ax1_twin, ax3_twin]:
    sns.despine(offset=3, trim=True, left=False, right=False, top=True, bottom=False, ax=this_ax)

plt.show()
if save_figs:
    f.savefig(os.path.join(outdir, '{0}_{1}.png'.format(file_str, 'rich-club')), dpi=300, bbox_inches='tight', pad_inches=0.01)

# Model performance vs. modularity

In [ ]:
# for binary graphs
modularity_directed = np.zeros((n_runs, n_logged_epochs))
participation_coefficient = np.zeros((n_runs, n_logged_epochs, np.sum(masks['bystanders'])))

for run in tqdm(np.arange(n_runs)):
    for epoch in np.arange(n_logged_epochs):
        A = hidden_weights[run, epoch].copy()

        # retain only bystanders
        A = A[masks['bystanders_bystanders']]
        A = A.reshape(np.sum(masks['bystanders']), np.sum(masks['bystanders']))

        A = np.abs(A)
        np.fill_diagonal(A, 0)
        thresh = np.quantile(A, q=0.9)
        mask = A > thresh
        A[mask] = 1
        A[~mask] = 0

        # for binary graphs
        ci, modularity_directed[run, epoch] = bct.algorithms.modularity_dir(A)
        participation_coefficient[run, epoch] = bct.algorithms.participation_coef(A, ci=ci, degree='undirected')

In [ ]:
fig_width = 8.5
fig_height = 3.5
f = plt.figure(layout=None, figsize=(fig_width, fig_height))
gs = f.add_gridspec(nrows=2, ncols=4, hspace=0.5, wspace=0.75)
# ax0_0 = f.add_subplot(gs[0, 0])
# ax0_1 = f.add_subplot(gs[0, 1])
# ax0_2 = f.add_subplot(gs[0, 2])
# ax0_3 = f.add_subplot(gs[0, 3])
ax1 = f.add_subplot(gs[0, :])
ax2 = f.add_subplot(gs[1, :])

# heatmaps
# indices = [3, 10, 50, 80]
# indices = [5, 15, 25, 45]
indices = np.linspace(2.5, n_logged_epochs-10, 4).astype(int)
n_indices = len(indices)
color_palette = sns.color_palette("mako", n_colors=n_runs)
# ax0_list = [ax0_0, ax0_1, ax0_2, ax0_3]
# for epoch in np.arange(n_indices):
#     ax0_list[epoch].set_title('Epoch: {0}'.format(indices[epoch]*100))
#
#     # A = hidden_weights[0, indices[epoch]].copy()
#     A = hidden_weights[:, indices[epoch]].copy()
#     A = np.abs(A)
#     if A.ndim == 3:
#         A = np.mean(A, axis=0)
#     A = A[masks['bystanders_bystanders']]
#     A = A.reshape(np.sum(masks['bystanders']), np.sum(masks['bystanders']))
#
#     ci, q = bct.algorithms.modularity_louvain_dir(A)
#     n_modules = ci.max()
#     ci_idx = np.argsort(ci)
#     ci_sorted = ci[ci_idx]
#     A = A[ci_idx, :]
#     A = A[:, ci_idx]
#     A = np.log10(A)
#     sns.heatmap(A, ax=ax0_list[epoch], square=True, cmap=color_palette, cbar_kws={'shrink': 0.5, 'label': 'log10(A)'})
#     # ax0_list[epoch].set_title(n_modules)
#     ax0_list[epoch].set_xticklabels('')
#     ax0_list[epoch].set_yticklabels('')
#     ax0_list[epoch].set_xlabel('node, i', labelpad=-3)
#     ax0_list[epoch].set_ylabel('node, j', labelpad=-3)
#
#     m_loc = 0
#     for m in np.arange(n_modules):
#         m_size = np.sum(ci_sorted == m+1)
#         ax0_list[epoch].add_patch(patches.Rectangle((m_loc, m_loc), m_size, m_size, edgecolor='w', fill=False, lw=1))
#         m_loc += m_size

color_palette = sns.color_palette("Set2")
for i, this_ax in enumerate([ax1, ax2]):
    this_ax.plot(x, accuracy_mean, color=color_palette[0], label='RNN-SA')
    this_ax.fill_between(x, accuracy_ci_lower, accuracy_ci_upper, color=color_palette[0], alpha=0.25)
    this_ax.set_ylim([-2.5, 100])
    this_ax.set_ylabel('Test Accuracy (%)')
    if i == 1:
        this_ax.set_xlabel('Epochs')
    # for epoch in np.arange(n_indices):
    #     this_ax.axvline(indices[epoch]*100, color='gray', linestyle='--')

    # Degree, trimmed mean
    if i == 0:
        y_mean = modularity_directed.mean(axis=0)[1:]
        y_std = modularity_directed.std(axis=0)[1:]
        label = 'Modularity'
        loc = 'lower right'
    elif i == 1:
        y_mean = participation_coefficient.mean(axis=-1).mean(axis=0)[1:]
        y_std = participation_coefficient.mean(axis=-1).std(axis=0)[1:]
        label = 'Participation Coefficient'
        loc = 'center right'
    ci = 1.96 * (y_std / np.sqrt(n_runs))
    ci_lower = y_mean - ci
    ci_upper = y_mean + ci

    ax_twin = this_ax.twinx()
    ax_twin.plot(x, y_mean, color='k', label=label)
    ax_twin.fill_between(x, ci_lower, ci_upper, color='k', alpha=0.25)
    ax_twin.set_ylabel(label)
    lines, labels = this_ax.get_legend_handles_labels()
    lines2, labels2 = ax_twin.get_legend_handles_labels()
    ax_twin.legend(lines + lines2, labels + labels2, loc=loc)

    sns.despine(offset=5, trim=True, left=False, right=False, top=True, bottom=False, ax=this_ax)
    sns.despine(offset=5, trim=True, left=False, right=False, top=True, bottom=False, ax=ax_twin)

plt.show()
if save_figs:
    f.savefig(os.path.join(outdir, '{0}_{1}.png'.format(file_str, 'modularity-pc')), dpi=300, bbox_inches='tight', pad_inches=0.01)